# 06 Biopython and ML-ready Sequence Features

This notebook introduces practical bioinformatics workflows and ML-ready biological sequence representation.

It supports:

- **Module 12: Practical Bioinformatics with Biopython**
- **Module 13: ML-ready Bioinformatics Bridge**

It connects selected Rosalind Armory-style problems such as `INI`, `FRMT`, `TFSQ`, `PHRE`, `FILT`, `BPHR`, and `ORFR` with feature extraction ideas from `GC`, `KMER`, `CONS`, `HAMM`, `EDIT`, and related sequence-analysis problems.

The goal is to show how biological sequence data can move from raw FASTA/FASTQ-style input to cleaned records, numerical features, feature tables, and simple baseline ML-style exploration.

## Learning Goals

After completing this notebook, a learner should be able to:

- read FASTA-style records using Biopython;
- understand basic sequence records;
- compute GC content from sequence records;
- translate DNA into protein;
- detect simple open reading frames;
- parse FASTQ-style quality scores;
- filter reads using quality thresholds;
- extract k-mer count features;
- build a machine-learning-ready feature matrix;
- apply a simple baseline clustering workflow;
- interpret the difference between bioinformatics preprocessing and machine learning.

## Connection to Original Rosalind Solutions

This notebook is connected to my original Rosalind solutions preserved under:

- `original_rosalind_tracks/bioinformatics_armory/`
- `original_rosalind_tracks/bioinformatics_stronghold/`
- `original_rosalind_tracks/bioinformatics_textbook_track/`

The notebook uses teaching-oriented implementations and small self-contained examples. The original solution files remain the solved-work archive, while this notebook reorganizes selected ideas for explanation, practical workflow design, and future reuse.

## 1. Imports

This notebook uses Biopython for sequence handling and common Python data-science libraries for feature tables and simple ML-style exploration.

In [ ]:
from io import StringIO
from itertools import product
from collections import Counter

import numpy as np
import pandas as pd

from Bio import SeqIO
from Bio.Seq import Seq
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

## 2. Reading FASTA records with Biopython

This connects to practical Rosalind Armory-style workflows such as `INI` and `FRMT`.

A FASTA record contains:

- an identifier line beginning with `>`;
- one or more sequence lines.

Biopython's `SeqIO.parse()` can read FASTA records from files or file-like objects.

In [ ]:
fasta_text = """>seq_1
ATGCGATACGCTTGA
>seq_2
ATGCCCGGGTTTAAA
>seq_3
TTTATGCGCGCGTAA
>seq_4
ATATATATATATATA
"""


records = list(SeqIO.parse(StringIO(fasta_text), "fasta"))

for record in records:
    print(record.id, record.seq, len(record.seq))

## 3. Sequence records as structured data

A Biopython sequence record contains both the biological sequence and associated metadata such as an ID or description.

For ML-ready workflows, these records often need to be converted into rows in a table.

In [ ]:
record_table = pd.DataFrame(
    {
        "record_id": [record.id for record in records],
        "sequence": [str(record.seq) for record in records],
        "length": [len(record.seq) for record in records],
    }
)

record_table

## 4. GC content as a numerical feature

This connects to Rosalind problem `GC`.

GC content is a simple but useful numerical feature for DNA sequences.

In [ ]:
def gc_content(sequence: str) -> float:
    """Return GC content percentage for a DNA sequence."""
    if not sequence:
        return 0.0

    gc_count = sequence.count("G") + sequence.count("C")
    return (gc_count / len(sequence)) * 100


record_table["gc_content"] = record_table["sequence"].apply(gc_content)
record_table

## 5. Transcription and translation

This connects to Rosalind problems such as `RNA`, `PROT`, `PTRA`, and `ORFR`.

Biopython's `Seq` object can transcribe DNA to RNA and translate nucleotide sequences into amino-acid sequences.

In [ ]:
dna = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")

rna = dna.transcribe()
protein = dna.translate(to_stop=True)

print("DNA:", dna)
print("RNA:", rna)
print("Protein:", protein)

## 6. Simple ORF detection

This connects to Rosalind problems such as `ORF` and `ORFR`.

An open reading frame often begins with a start codon `ATG` and ends at a stop codon. The implementation below is intentionally simple and checks the three forward reading frames.

In [ ]:
STOP_CODONS = {"TAA", "TAG", "TGA"}


def find_forward_orfs(sequence: str) -> list[tuple[int, int, str]]:
    """Find simple forward-strand ORFs and return start, end, and protein."""
    orfs = []

    for frame in range(3):
        for start in range(frame, len(sequence) - 2, 3):
            codon = sequence[start:start + 3]

            if codon != "ATG":
                continue

            for end in range(start + 3, len(sequence) - 2, 3):
                stop_codon = sequence[end:end + 3]

                if stop_codon in STOP_CODONS:
                    coding_sequence = sequence[start:end + 3]
                    protein = str(Seq(coding_sequence).translate(to_stop=True))
                    orfs.append((start, end + 3, protein))
                    break

    return orfs


find_forward_orfs("CCCATGGCCAAATAACCCATGTTTTAG")

## 7. FASTQ records and quality scores

This connects to Rosalind Armory-style problems such as `TFSQ`, `PHRE`, `FILT`, and `BPHR`.

FASTQ records include both sequences and quality scores. Quality scores estimate confidence in each base call.

In [ ]:
fastq_text = """@read_1
ACGTACGT
+
IIIIIIII
@read_2
ACGTTCGT
+
!!!!IIII
@read_3
GGGGCCCC
+
IIII!!!!
"""


fastq_records = list(SeqIO.parse(StringIO(fastq_text), "fastq"))

for record in fastq_records:
    print(record.id)
    print(record.seq)
    print(record.letter_annotations["phred_quality"])

## 8. Average read quality

A simple quality-control step is to compute the average Phred quality score for each read.

In [ ]:
def average_quality(record) -> float:
    """Return the average Phred quality score for a FASTQ record."""
    qualities = record.letter_annotations["phred_quality"]

    if not qualities:
        return 0.0

    return sum(qualities) / len(qualities)


for record in fastq_records:
    print(record.id, average_quality(record))

## 9. Filtering reads by quality

This connects to Rosalind problems such as `FILT` and `BPHR`.

The function below keeps only reads with average quality greater than or equal to a threshold.

In [ ]:
def filter_reads_by_average_quality(records, threshold: float):
    """Return FASTQ records whose average quality is at least the threshold."""
    return [
        record
        for record in records
        if average_quality(record) >= threshold
    ]


filtered_records = filter_reads_by_average_quality(fastq_records, threshold=30)

for record in filtered_records:
    print(record.id, record.seq, average_quality(record))

## 10. k-mer count features

This connects to Rosalind problems such as `KMER` and Textbook Track problems such as `BA1A` and `BA1B`.

k-mer features are useful because they convert variable biological sequences into numerical summaries.

In [ ]:
def kmer_counts(sequence: str, k: int) -> dict[str, int]:
    """Return observed k-mer counts for a sequence."""
    counts = Counter()

    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i + k]
        counts[kmer] += 1

    return dict(counts)


def all_dna_kmers(k: int) -> list[str]:
    """Return all DNA k-mers in lexicographic order."""
    return ["".join(kmer) for kmer in product("ACGT", repeat=k)]


kmer_counts("ACGTACGT", k=2)

## 11. Building a feature matrix

A feature matrix is a table where:

- each row is a sequence;
- each column is a numerical feature.

Here we combine length, GC content, and 2-mer counts.

In [ ]:
def build_sequence_feature_matrix(records, k: int = 2) -> pd.DataFrame:
    """Build a feature matrix from Biopython sequence records."""
    vocabulary = all_dna_kmers(k)

    rows = []
    index = []

    for record in records:
        sequence = str(record.seq)
        counts = kmer_counts(sequence, k)

        row = {
            "length": len(sequence),
            "gc_content": gc_content(sequence),
        }

        for kmer in vocabulary:
            row[f"kmer_{kmer}"] = counts.get(kmer, 0)

        rows.append(row)
        index.append(record.id)

    return pd.DataFrame(rows, index=index)


features = build_sequence_feature_matrix(records, k=2)
features

## 12. Scaling features

Machine learning workflows often scale numerical features before clustering or classification.

Scaling makes features comparable when they are on different numerical ranges.

In [ ]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

pd.DataFrame(
    scaled_features,
    index=features.index,
    columns=features.columns,
)

## 13. Baseline clustering

This is a simple ML-style demonstration, not a biological claim.

We use k-means clustering to group sequences based on length, GC content, and k-mer counts.

In [ ]:
model = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = model.fit_predict(scaled_features)

clustered_table = features.copy()
clustered_table["cluster"] = clusters

clustered_table

## 14. Interpreting feature-based outputs

The clusters above are based only on simple engineered features.

A careful interpretation should ask:

- Which features were used?
- Are the sequences long enough for the features to be meaningful?
- Are the labels or clusters biologically validated?
- Is the dataset large enough?
- Are we using this as exploration or as a trained predictive model?

This distinction is important when moving from computational exercises to real biological data science.

## 15. Mini exercise set

Try modifying the functions above to solve these small exercises.

1. Add AT content as a new feature.
2. Change the k-mer size from 2 to 3.
3. Normalize k-mer counts by sequence length.
4. Filter FASTQ reads by minimum base quality instead of average quality.
5. Add reverse-complement features to the feature table.
6. Try clustering with three clusters instead of two.
7. Explain why clustering output should not be treated as biological truth without validation.

## Summary

This notebook showed how practical bioinformatics workflows can connect to ML-ready biological sequence representation.

| Step | Purpose |
|---|---|
| FASTA parsing | read sequence records |
| Sequence tables | organize biological data |
| GC content | create composition features |
| Translation and ORF detection | connect sequence data to protein-level interpretation |
| FASTQ parsing | include quality-aware sequencing data |
| Quality filtering | clean biological inputs |
| k-mer counts | build sequence feature vectors |
| Feature matrix | prepare data for ML workflows |
| Scaling and clustering | demonstrate baseline exploratory analysis |

The key message is that ML-ready bioinformatics starts with careful biological preprocessing, transparent feature construction, and cautious interpretation.